In [43]:
import pandas as pd
import io
coffee_customers= pd.read_csv("coffee_customers.csv")

In [44]:
coffee_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 35166 entries, 0 to 35165
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   YTD_Avg_Order_Size          35166 non-null  float64
 1   Avg_Monthly_Points_Expired  35166 non-null  float64
 2   Customer_Age                35166 non-null  int64  
 3   Loyalty_Tier                34465 non-null  str    
 4   Annual_Spend_Bracket        35166 non-null  str    
 5   Visit_Time                  35166 non-null  str    
 6   Primary_Purchase            34463 non-null  str    
 7   Order_Type                  35166 non-null  str    
 8   Dairy_Preference            35166 non-null  str    
 9   App_User                    35166 non-null  str    
 10  Annual_Visit_Frequency      35166 non-null  int64  
 11  Store_Location              34977 non-null  str    
 12  Will_Return_Next_Week       11722 non-null  str    
dtypes: float64(2), int64(2), str(9)
memory usa

In [45]:
coffee_labeled= coffee_customers[coffee_customers['Will_Return_Next_Week'].notnull()].copy()
coffee_unlabeled = coffee_customers[coffee_customers['Will_Return_Next_Week'].isnull()].copy()

coffee_labeled.head()

,YTD_Avg_Order_Size,Avg_Monthly_Points_Expired,Customer_Age,Loyalty_Tier,Annual_Spend_Bracket,Visit_Time,Primary_Purchase,Order_Type,Dairy_Preference,App_User,Annual_Visit_Frequency,Store_Location,Will_Return_Next_Week
0,1.000000,21.65978,36,Gold Member,$501 - $750,Morning-Peak,Cold Brew,Dine-in,Regular,Registered-Mobile,50,Arts District Flagship,Yes
1,1.000000,0.00000,36,NaN,$301 - $400,Afternoon,NaN,Drive-Thru,Regular,Registered-Mobile,40,Arts District Flagship,No
2,1.291923,0.00000,36,Gold Member,$1001 - $1500,Morning-Peak,Pour-over,Dine-in,Regular,Registered-Mobile,40,Arts District Flagship,Yes
3,1.000000,0.00000,22,Gold Member,$301 - $400,Afternoon,Nitro,Drive-Thru,Regular,Registered-Mobile,40,Arts District Flagship,No
4,1.000000,0.00000,43,Gold Member,$301 - $400,Evening-Early,Espresso,Mobile-Order,Oat,Registered-Mobile,40,Arts District Flagship,No


In [46]:
coffee_unlabeled.head()

,YTD_Avg_Order_Size,Avg_Monthly_Points_Expired,Customer_Age,Loyalty_Tier,Annual_Spend_Bracket,Visit_Time,Primary_Purchase,Order_Type,Dairy_Preference,App_User,Annual_Visit_Frequency,Store_Location,Will_Return_Next_Week
11722,1.0,0.0,46,Gold Member,$51 - $75,Evening-Early,Latte,Takeout,Regular,In-Store Only,40,Arts District Flagship,NaN
11723,1.0,0.0,20,Gold Member,$401 - $500,Afternoon,Flat White,Takeout,Regular,In-Store Only,40,Arts District Flagship,NaN
11724,1.0,0.0,22,Gold Member,$501 - $750,Afternoon,Latte,Mobile-Order,Oat,Registered-Mobile,52,Arts District Flagship,NaN
11725,1.0,0.0,35,Gold Member,$301 - $400,Morning-Peak,Americano,Dine-in,Oat,Registered-Mobile,40,Arts District Flagship,NaN
11726,1.0,0.0,40,Gold Member,$751 - $1000,Evening-Early,Americano,Delivery-App,Regular,In-Store Only,40,Arts District Flagship,NaN


In [47]:
from sklearn.model_selection import train_test_split

coffee_labeled_training, coffee_validation = train_test_split(coffee_labeled,
                                  test_size=0.40, random_state=42)


In [48]:
coffee_training= pd.concat([coffee_labeled_training,coffee_unlabeled])

In [49]:
coffee_training.head()

,YTD_Avg_Order_Size,Avg_Monthly_Points_Expired,Customer_Age,Loyalty_Tier,Annual_Spend_Bracket,Visit_Time,Primary_Purchase,Order_Type,Dairy_Preference,App_User,Annual_Visit_Frequency,Store_Location,Will_Return_Next_Week
6725,1.000000,0.0,49,Gold Member,$401 - $500,Morning-Peak,Cortado,Dine-in,Regular,Registered-Mobile,40,Arts District Flagship,Yes
8122,1.000000,0.0,28,Gold Member,$151 - $200,Morning-Peak,Iced Coffee,Drive-Thru,Regular,Registered-Mobile,40,Arts District Flagship,No
2729,1.344563,0.0,39,Gold Member,$1501 - $2000,Afternoon,Cold Brew,Takeout,Oat,Registered-Mobile,52,Arts District Flagship,Yes
11602,1.000000,0.0,33,NaN,$301 - $400,Morning-Peak,NaN,Drive-Thru,Regular,In-Store Only,20,Arts District Flagship,No
433,1.000000,0.0,41,Gold Member,$1001 - $1500,Afternoon,Drip Coffee,Takeout,Regular,Registered-Mobile,52,Arts District Flagship,No


In [50]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
import numpy as np

coffee_training = coffee_training.replace('?', np.nan)

ord_features = ['Loyalty_Tier', 'Annual_Spend_Bracket']
nom_features= ['Visit_Time','Primary_Purchase','Order_Type', 'Dairy_Preference',
                 'App_User','Store_Location']
num_features= ['YTD_Avg_Order_Size', 'Avg_Monthly_Points_Expired', 'Customer_Age', 'Annual_Visit_Frequency']

loyalty_order = ['Unknown','Window Shopper', 'First-time Visitor', 'Guest',
                 'Occasional', 'Silver Member', 'Gold Member', 'Platinum Member', 'VIP']

spend_order = ['$0 - $10', '$11 - $25','$26 - $50','$51 - $75','$76 - $100',
               '$101 - $150','$151 - $200','$201 - $300','$301 - $400','$401 - $500','$501 - $750',
               '$751 - $1000','$1001 - $1500','$1501 - $2000','$2001 - $3000',
               '$3000+']

nom_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value= 'Unknown'),
    OneHotEncoder(handle_unknown='ignore',sparse_output=False)
)

ord_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='Unknown'),
    OrdinalEncoder(categories=[loyalty_order, spend_order])
)

num_pipe = make_pipeline(
    SimpleImputer(strategy='median')
)

preprocessing = ColumnTransformer([
    ('ord', ord_pipe, ord_features ),
    ('nom', nom_pipe, nom_features),
    ('num', num_pipe, num_features)
])

In [51]:
X_train = coffee_training.drop(columns=['Will_Return_Next_Week']).copy()
y_train = coffee_training['Will_Return_Next_Week'].copy()

X_train_processed = preprocessing.fit_transform(X_train)

from sklearn.cluster import HDBSCAN
hdb = HDBSCAN(
    min_cluster_size=5,
    min_samples=50,
    cluster_selection_epsilon=0.1,
    copy=True
)
clusters = hdb.fit_predict(X_train_processed)


In [52]:
# add temporary column to coffee_training to prune dataset
coffee_training['cluster_id'] = clusters
#drop instances where cluster is -1
coffee_training_filtered = coffee_training[coffee_training['cluster_id'] != -1].copy()
#remove temporary column clusters
X_filtered = coffee_training_filtered.drop(columns=['Will_Return_Next_Week', 'cluster_id'])
y_filtered = coffee_training_filtered['Will_Return_Next_Week']

In [53]:
from sklearn.semi_supervised import LabelSpreading

X_filtered_processed = preprocessing.transform(X_filtered)
#map yes to 1 and No to 0, fill in Nan's with -1
y_prepared = y_filtered.map({'Yes': 1, 'No': 0}).fillna(-1).astype(int)
ls_model = LabelSpreading(kernel='knn', n_neighbors=35,alpha=0.2)
ls_model.fit(X_filtered_processed, y_prepared)
#g sore label spreading transductuons attribute
y_spread = pd.Series(ls_model.transduction_, index=y_filtered.index)

In [54]:
y_spread.value_counts()

0    23404
1     6048
Name: count, dtype: int64

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA, FastICA
import pandas as pd

#PCA
pca_pipeline = Pipeline([
    ('prep', preprocessing),
    ('pca', PCA(n_components=5))
])
#ICA
ica_pipeline = Pipeline([
    ('prep', preprocessing),
    ('ica', FastICA(n_components=5, random_state=42, max_iter=1000))
])
#fit pipeline
X_pca = pca_pipeline.fit_transform(X_filtered)
X_ica= ica_pipeline.fit_transform(X_filtered)
#convert to df
X_pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(5)])
X_ica_df = pd.DataFrame(X_ica, columns=[f'ICA{i+1}' for i in range(5)])


In [56]:
# X_pca_df.head()
X_ica_df.head()

,ICA1,ICA2,ICA3,ICA4,ICA5
0,-0.132127,0.183531,0.406151,-0.192957,0.794140
1,-1.159988,0.174417,0.261417,0.205161,-0.850444
2,1.475609,0.190183,0.431294,0.854036,0.242569
3,-0.070812,0.137180,-3.390545,-1.268523,-0.478539
4,1.058437,0.191287,0.414204,0.869848,0.364089


In [57]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

#Model A
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_filtered_processed, y_spread)
#Model B
ada_model = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_model.fit(X_filtered_processed, y_spread)

,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",100
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels.","ndarray[int64](2,)","[0,1]"
estimator_ estimator_: estimatorThe base estimator from which the ensemble is grown... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,DecisionTreeClassifier,DecisionTreeC...r(max_depth=1)
estimator_errors_ estimator_errors_: ndarray of floatsClassification error for each estimator in the boostedensemble.,"ndarray[float64](100,)","[0.21,0.26,0.24,...,0.5 ,0.5 ,0.49]"
estimator_weights_ estimator_weights_: ndarray of floatsWeights for each estimator in the boosted ensemble.,"ndarray[float64](100,)","[1.35,1.06,1.15,...,0.01,0.01,0.02]"
estimators_ estimators_: list of classifiersThe collection of fitted sub-estimators.,list,"[DecisionTreeC...te=1608637542), DecisionTreeC...te=1273642419), DecisionTreeC...te=1935803228), DecisionTreeC...ate=787846414), ...]"
"feature_importances_ feature_importances_: ndarray of shape (n_features,)The impurity-based feature importances if supported by the``estimator`` (when based on decision trees).Warning: impurity-based feature importances can be misleading forhigh cardinality features (many unique values). See:func:`sklearn.inspection.permutation_importance` as an alternative.","ndarray[float64](83,)","[0.01,0.29,0.01,...,0.07,0.22,0.18]"


In [58]:
from sklearn.pipeline import Pipeline
#MODEL 1A --> PCA + Random Forest
pipe_1A = Pipeline([
    ('prep', preprocessing),
    ('pca', PCA(n_components=5)),
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

#MODEL 1B --> PCA + AdaBoost
pipe_1B = Pipeline([
    ('prep', preprocessing),
    ('pca', PCA(n_components=5)),
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=42))
])

#MODEL 2A --> FastICA + Random Forest
pipe_2A = Pipeline([
    ('prep', preprocessing),
    ('ica', FastICA(n_components=5, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

#MODEL 2B --> FastICA + AdaBoost
pipe_2B = Pipeline([
    ('prep', preprocessing),
    ('ica', FastICA(n_components=5, random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=100, random_state=42))
])

# train all 4 on the filtered data
pipelines = [pipe_1A, pipe_1B, pipe_2A, pipe_2B]
names = ["1A", "1B", "2A", "2B"]
y_spread = y_spread.map({0: 'No', 1: 'Yes'})
for pipe, name in zip(pipelines, names):
    print(f"Training Pipeline {name}...")
    pipe.fit(X_filtered, y_spread)

Training Pipeline 1A...
Training Pipeline 1B...
Training Pipeline 2A...
Training Pipeline 2B...


In [59]:
X_test = coffee_validation.drop(columns=['Will_Return_Next_Week'])
y_test = coffee_validation['Will_Return_Next_Week']

In [61]:
# Pass the raw, un-encoded dataframe to the preprocessor
X_test = X_test.replace('?', np.nan)
X_test_processed = preprocessing.transform(X_test)

In [64]:
from sklearn.metrics import classification_report

#convert the answers from "Yes/No" to 1s and 0s to match predictions
y_test_prepared = y_test.map({'Yes': 1, 'No': 0}).astype(int)

#Run the evaluation loop on the cleaned data
for pipe in pipelines:
    
    #grab the name of the final classifier in the pipeline
    model_name = pipe.steps[-1][1].__class__.__name__
    
    print(f"--- {model_name} ---")
    
    #predict using processed features
    y_pred = pipe.predict(X_test)
    
    # Compare against prepared answers
    print(classification_report(y_test, y_pred))
    print("\n" + "="*50 + "\n")

--- RandomForestClassifier ---
              precision    recall  f1-score   support

          No       0.87      0.87      0.87      3571
         Yes       0.58      0.57      0.57      1118

    accuracy                           0.80      4689
   macro avg       0.72      0.72      0.72      4689
weighted avg       0.80      0.80      0.80      4689



--- AdaBoostClassifier ---
              precision    recall  f1-score   support

          No       0.85      0.92      0.88      3571
         Yes       0.64      0.47      0.54      1118

    accuracy                           0.81      4689
   macro avg       0.74      0.69      0.71      4689
weighted avg       0.80      0.81      0.80      4689



--- RandomForestClassifier ---
              precision    recall  f1-score   support

          No       0.87      0.88      0.87      3571
         Yes       0.60      0.58      0.59      1118

    accuracy                           0.81      4689
   macro avg       0.73      0.73  